In [17]:
!pip -q install rpy2
%load_ext rpy2.ipython


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [18]:
%%R
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate, stringr, readxl, data.table, gdata, scales)

source("functions-1.R")
source("rating_variables.R")
monthlist <- sprintf("%02d", 1:12)
y <- 2010

# Plan (enrollment and contract) data

In [19]:
%%R

plan.data <- map_dfr(monthlist, ~ load_month(.x, y)) %>%
  arrange(contractid, planid, state, county, month) %>%
  group_by(state, county) %>%
  fill(fips, .direction = "downup") %>%
  ungroup() %>%
  group_by(contractid, planid) %>%
  fill(plan_type, partd, snp, eghp, plan_name, .direction = "downup") %>%
  ungroup() %>%
  group_by(contractid) %>%
  fill(org_type, org_name, org_marketing_name, parent_org, .direction = "downup") %>%
  ungroup()

plan.data.dt <- as.data.table(plan.data)
setorder(plan.data.dt, contractid, planid, fips, year, month)

plan.year <- plan.data.dt[
, {
    nonmiss <- !is.na(enrollment)
    n <- sum(nonmiss)
    list(
      n_nonmiss = n,
      avg_enrollment = if (n>0) mean(enrollment[nonmiss]) else NA_real_,
      sd_enrollment  = if (n>1) sd(enrollment[nonmiss]) else NA_real_,
      min_enrollment = if (n>0) min(enrollment[nonmiss]) else NA_real_,
      max_enrollment = if (n>0) max(enrollment[nonmiss]) else NA_real_,
      first_enrollment = if (n>0) enrollment[which(nonmiss)[1]] else NA_real_,
      last_enrollment  = if (n>0) enrollment[tail(which(nonmiss), 1)] else NA_real_,
      state  = tail(state, 1),
      county = tail(county, 1),
      org_type = tail(org_type, 1),
      plan_type = tail(plan_type, 1),
      partd = tail(partd, 1),
      snp   = tail(snp, 1),
      eghp  = tail(eghp, 1),
      org_name = tail(org_name, 1),
      org_marketing_name = tail(org_marketing_name, 1),
      plan_name = tail(plan_name, 1),
      parent_org = tail(parent_org, 1),
      contract_date = tail(contract_date, 1)
    )
  },
by = .(contractid, planid, fips, year)
]

plan.data.year <- as_tibble(plan.year)

In addition: There were 12 warnings (use warnings() to see them)


# Service area data

In [20]:
%%R

monthlist_subset <- sprintf("%02d", 1:11)
service.year <- map_dfr(monthlist_subset, ~ load_month_sa(.x, y))

service.year <- service.year %>%
  arrange(contractid, fips, state, county, month)

service.year <- service.year %>%
  group_by(state, county) %>%
  fill(fips, .direction = "downup") %>%
  ungroup() %>%
  group_by(contractid) %>%
  fill(plan_type, partial, eghp, org_type, org_name, .direction = "downup") %>%
  ungroup()

service.data.year <- service.year %>%
  group_by(contractid, fips, year) %>%
  arrange(month, .by_group = TRUE) %>%
  summarize(
    state     = last(state),
    county    = last(county),
    org_name  = last(org_name),
    org_type  = last(org_type),
    plan_type = last(plan_type),
    partial   = last(partial),
    eghp      = last(eghp),
    ssa       = last(ssa),
    notes     = last(notes),
    .groups = "drop"
  )

In addition: There were 11 warnings (use warnings() to see them)


# Penetration data

In [21]:
%%R
ma.penetration <- map_dfr(monthlist, ~ load_month_pen(.x, y)) %>%
  arrange(state, county, month) %>%
  group_by(state, county) %>%
  fill(fips, .direction = "downup") %>%
  ungroup()

pen.year <- ma.penetration %>%
  group_by(fips, state, county, year) %>%
  arrange(month, .by_group = TRUE) %>%
  summarize(
    n_elig  = sum(!is.na(eligibles)),
    n_enrol = sum(!is.na(enrolled)),

    avg_eligibles   = ifelse(n_elig  > 0, mean(eligibles, na.rm = TRUE), NA_real_),
    sd_eligibles    = ifelse(n_elig  > 1,  sd(eligibles,  na.rm = TRUE), NA_real_),
    min_eligibles   = ifelse(n_elig  > 0, min(eligibles,  na.rm = TRUE), NA_real_),
    max_eligibles   = ifelse(n_elig  > 0, max(eligibles,  na.rm = TRUE), NA_real_),
    first_eligibles = ifelse(n_elig  > 0, first(na.omit(eligibles)),     NA_real_),
    last_eligibles  = ifelse(n_elig  > 0,  last(na.omit(eligibles)),     NA_real_),

    avg_enrolled    = ifelse(n_enrol > 0, mean(enrolled,   na.rm = TRUE), NA_real_),
    sd_enrolled     = ifelse(n_enrol > 1,  sd(enrolled,    na.rm = TRUE), NA_real_),
    min_enrolled    = ifelse(n_enrol > 0, min(enrolled,    na.rm = TRUE), NA_real_),
    max_enrolled    = ifelse(n_enrol > 0, max(enrolled,    na.rm = TRUE), NA_real_),
    first_enrolled  = ifelse(n_enrol > 0, first(na.omit(enrolled)),       NA_real_),
    last_enrolled   = ifelse(n_enrol > 0,  last(na.omit(enrolled)),       NA_real_),

    ssa = last(ssa),
    .groups = "drop"
  )

In addition: Warning messages:
1: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 
2: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 
3: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 


# Star ratings data

In [22]:
%%R
library(tidyverse)

# Load 2010 built file
d2010 <- read_csv("../data/data-2010.csv", show_col_types = FALSE)

# Component measures used to build a continuous raw score
rating_vars_2010 <- c(
  "breastcancer_screen","rectalcancer_screen","cv_diab_cholscreen","glaucoma_test","monitoring",
  "flu_vaccine","pn_vaccine","physical_health","mental_health","osteo_test","physical_monitor",
  "primaryaccess","osteo_manage","diab_healthy","bloodpressure","ra_manage","copd_test",
  "bladder","falling","nodelays","doctor_communicate","carequickly","customer_service",
  "overallrating_care","overallrating_plan","complaints_plan","appeals_timely","appeals_review",
  "leave_plan","audit_problems","hold_times","info_accuracy","ttyt_available"
)

# Construct continuous running variable
d2010 <- d2010 %>%
  mutate(
    raw_rating = rowMeans(across(all_of(rating_vars_2010)), na.rm = TRUE),
    raw_rating = if_else(is.nan(raw_rating), NA_real_, raw_rating)
  )

# Plan-level file for RD (one row per plan)
plans2010 <- d2010 %>%
  group_by(contractid, planid) %>%
  summarise(
    raw_rating = mean(raw_rating, na.rm = TRUE),
    Star_Rating = first(na.omit(Star_Rating)),
    enrollments = sum(avg_enrollment, na.rm = TRUE),
    partd = first(na.omit(partd)),
    plan_type = first(na.omit(plan_type)),
    .groups = "drop"
  ) %>%
  mutate(raw_rating = if_else(is.nan(raw_rating), NA_real_, raw_rating))

# Sanity check (should NOT be 1 anymore)
plans2010 %>%
  summarise(
    n = n(),
    n_raw = sum(!is.na(raw_rating)),
    share_on_half_grid = mean(abs(raw_rating*2 - round(raw_rating*2)) < 1e-8, na.rm = TRUE),
    match_rounded_star = mean(round(raw_rating*2)/2 == Star_Rating, na.rm = TRUE)
  )



# A tibble: 1 × 4
      n n_raw share_on_half_grid match_rounded_star
  <int> <int>              <dbl>              <dbl>
1  1563  1563             0.0377              0.836


In addition: Warning message:
Returning more (or less) than 1 row per `summarise()` group was deprecated in
dplyr 1.1.0.
ℹ Please use `reframe()` instead.
ℹ When switching from `summarise()` to `reframe()`, remember that `reframe()`
  always returns an ungrouped data frame and adjust accordingly.
Call `lifecycle::last_lifecycle_warnings()` to see where this warning was
generated. 


In [23]:
%%R
library(tidyverse)

first_nonmiss <- function(x) dplyr::first(x[!is.na(x)], default = NA)

plans2010 <- d2010 %>%
  group_by(contractid, planid) %>%
  summarise(
    raw_rating  = mean(raw_rating, na.rm = TRUE),
    Star_Rating = first_nonmiss(Star_Rating),
    enrollments = sum(avg_enrollment, na.rm = TRUE),
    partd       = first_nonmiss(partd),
    plan_type   = first_nonmiss(plan_type),
    .groups = "drop"
  ) %>%
  mutate(raw_rating = if_else(is.nan(raw_rating), NA_real_, raw_rating))


In [24]:
%%R
glimpse(plans2010)
summary(plans2010$raw_rating)
table(plans2010$Star_Rating, useNA = "ifany")


Rows: 2,527
Columns: 7
$ contractid  <chr> "H0084", "H0104", "H0104", "H0104", "H0105", "H0105", "H01…
$ planid      <dbl> 1, 2, 4, 8, 1, 2, 4, 5, 6, 5, 1, 10, 12, 21, 1, 23, 1, 8, …
$ raw_rating  <dbl> NA, 3.107143, 3.107143, 3.107143, NA, NA, NA, NA, NA, 2.15…
$ Star_Rating <dbl> NA, 3.0, 3.0, 3.0, NA, NA, NA, NA, NA, 2.0, 3.0, 3.0, 3.0,…
$ enrollments <dbl> 759.92027, 6601.28586, 30739.56061, 16336.15152, 52.41667,…
$ partd       <chr> "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Y…
$ plan_type   <chr> "Local PPO", "Local PPO", "Local PPO", "Local PPO", "Natio…

   2  2.5    3  3.5    4  4.5    5 <NA> 
  35  321  474  320  239  166    8  964 


# Benchmark data

In [25]:
%%R
bench.data <- read_csv("../../ma-data/ma/benchmarks/ratebook2010/CountyRate2010.csv",
                       skip = 10,
                       col_names = c("ssa","state","county_name","aged_parta",
                                     "aged_partb","disabled_parta","disabled_partb",
                                     "esrd_ab","risk_ab"),
                       show_col_types = FALSE, progress = FALSE)

bench.data.year <- bench.data %>%
  select(ssa, aged_parta, aged_partb, risk_ab) %>%
  mutate(ssa = as.numeric(ssa),
         risk_star5 = NA_real_, risk_star45 = NA_real_, risk_star4 = NA_real_,
         risk_star35 = NA_real_, risk_star3 = NA_real_, risk_star25 = NA_real_,
         risk_bonus5 = NA_real_, risk_bonus35 = NA_real_, risk_bonus0 = NA_real_,
         year = 2010)

# Merge data

In [27]:
%%R
rating_vars_2010 <- c(
  "breastcancer_screen","rectalcancer_screen","cv_diab_cholscreen","glaucoma_test","monitoring",
  "flu_vaccine","pn_vaccine","physical_health","mental_health","osteo_test","physical_monitor",
  "primaryaccess","osteo_manage","diab_healthy","bloodpressure","ra_manage","copd_test",
  "bladder","falling","nodelays","doctor_communicate","carequickly","customer_service",
  "overallrating_care","overallrating_plan","complaints_plan","appeals_timely","appeals_review",
  "leave_plan","audit_problems","hold_times","info_accuracy","ttyt_available"
)

ma.data <- plan.data.year %>%
  inner_join(service.data.year %>% select(contractid, fips), by = c("contractid","fips")) %>%
  filter(!state %in% c("VI","PR","MP","GU","AS",""),
         snp == "No",
         (planid < 800 | planid >= 900),
         !is.na(planid), !is.na(fips)) %>%
  left_join(
    pen.year %>% ungroup() %>%
      rename(state_long = state, county_long = county) %>%
      mutate(state_long = str_to_lower(state_long)) %>%
      group_by(fips) %>% mutate(ncount = n()) %>% filter(ncount == 1),
    by = c("fips")
  ) %>%
  left_join(
    star.data.year %>%
      distinct(contractid, .keep_all = TRUE) %>%
      select(-any_of(c("contract_name","org_type","org_marketing"))),
    by = c("contractid")
  ) %>%
  mutate(
    Star_Rating = case_when(
      partd == "No" ~ partc_score,
      partd == "Yes" & is.na(partcd_score) ~ partc_score,
      partd == "Yes" & !is.na(partcd_score) ~ partcd_score
    )
  ) %>%
  {
    d <- .
    d <- d %>%
      mutate(
        raw_proxy = rowMeans(pick(all_of(rating_vars_2010)), na.rm = TRUE),
        raw_proxy = if_else(is.nan(raw_proxy), NA_real_, raw_proxy)
      )
    if ("raw_rating" %in% names(d)) {
      d <- d %>% mutate(raw_rating = coalesce(raw_rating, raw_proxy))
    } else {
      d <- d %>% mutate(raw_rating = raw_proxy)
    }
    d %>% select(-raw_proxy)
  } %>%
  left_join(bench.data.year %>% filter(!is.na(ssa)), by = c("ssa")) %>%
  select(-starts_with("year.")) %>%
  mutate(year = y) %>%
  mutate(
    ma_rate = case_when(
      year < 2012 ~ risk_ab,
      year >= 2012 & year < 2015 & Star_Rating == 5    ~ risk_star5,
      year >= 2012 & year < 2015 & Star_Rating == 4.5  ~ risk_star45,
      year >= 2012 & year < 2015 & Star_Rating == 4    ~ risk_star4,
      year >= 2012 & year < 2015 & Star_Rating == 3.5  ~ risk_star35,
      year >= 2012 & year < 2015 & Star_Rating == 3    ~ risk_star3,
      year >= 2012 & year < 2015 & Star_Rating < 3     ~ risk_star25,
      year >= 2012 & year < 2015 & is.na(Star_Rating)  ~ risk_star35,
      year >= 2015 & Star_Rating >= 4                  ~ risk_bonus5,
      year >= 2015 & Star_Rating < 4                   ~ risk_bonus0,
      year >= 2015 & is.na(Star_Rating)                ~ risk_bonus35
    )
  )

write_csv(ma.data, paste0("../data/data-", y, ".csv"))


In [28]:
%%R
d2010 <- read_csv("../data/data-2010.csv", show_col_types = FALSE)
d2010 %>% summarise(
  has_raw_rating = "raw_rating" %in% names(d2010),
  share_on_half_grid = mean(abs(raw_rating*2 - round(raw_rating*2)) < 1e-8, na.rm = TRUE)
)

# A tibble: 1 × 2
  has_raw_rating share_on_half_grid
  <lgl>                       <dbl>
1 TRUE                       0.0395
